# Module 7 • Hugging Face and Pretrained Transformer Workflows

# Lesson 39 • Text Generation with Pretrained Language Models

**Course:** Natural Language Processing: From Foundations to Large Language Models  
**Author:** Eman Khater  
**Difficulty:** Intermediate  
**Estimated study time:** 180–220 minutes  
**Execution target:** CPU by default

---

## Scope

This lesson develops practical text generation with pretrained causal language
models. It covers prompt construction, causal generation, greedy decoding,
temperature, top-k sampling, nucleus sampling, repetition control, stopping
criteria, sequence scoring, diversity, hallucination risk, and responsible use.

The notebook contains:

1. a complete offline CPU experiment with a compact decoder-only Transformer;
2. optional Hugging Face tokenizer, AutoModelForCausalLM, pipeline, and generation
   cells that remain disabled by default.

## Learning Objectives

After completing this lesson, the learner should be able to:

- explain causal language modeling;
- distinguish prompting from fine-tuning;
- inspect token probabilities during generation;
- implement greedy decoding;
- control randomness with temperature;
- implement top-k and top-p sampling;
- apply repetition penalties;
- define stopping criteria;
- compare decoding strategies;
- calculate perplexity and diversity indicators;
- identify degeneration and hallucination risks;
- structure Hugging Face generation workflows;
- evaluate Arabic and multilingual generation considerations.

## Table of Contents

1. Pretrained Language Models
2. Causal Language Modeling
3. Prompting
4. Generation Loop
5. Greedy Decoding
6. Temperature
7. Top-k Sampling
8. Top-p Sampling
9. Repetition Penalties
10. Stopping Criteria
11. Offline Command Corpus
12. Train, Validation, and Test Splits
13. Tokenization and Vocabulary
14. Dataset and Dynamic Padding
15. Decoder-Only Transformer
16. Shape and Mask Inspection
17. Training Utilities
18. Model Training
19. Learning Curves
20. Validation Perplexity
21. Next-Token Prediction
22. Greedy Generation
23. Temperature Sampling
24. Top-k Sampling
25. Nucleus Sampling
26. Repetition Control
27. Stopping Rules
28. Strategy Comparison
29. Diversity Metrics
30. Length Analysis
31. Confidence and Sequence Scores
32. Failure Analysis
33. Hallucination and Grounding
34. Prompt Injection and Unsafe Instructions
35. Optional Hugging Face Setup
36. Optional AutoTokenizer
37. Optional AutoModelForCausalLM
38. Optional Text-Generation Pipeline
39. GenerationConfig
40. Arabic and Multilingual Considerations
41. Reproducibility and Reporting
42. Knowledge Check
43. Exercises
44. Summary and Next Lesson

# 1. Pretrained Language Models

A pretrained language model learns broad statistical patterns from large text
collections before being adapted or prompted for downstream tasks.

In [ ]:
import copy
import importlib.util
import json
import math
import platform
import random
import re
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.nn.utils import clip_grad_norm_
from torch.utils.data import DataLoader, Dataset

model_uses = pd.DataFrame(
    [
        ('Prompting', 'no parameter update'),
        ('Fine-tuning', 'update model parameters'),
        ('Adapters', 'update small added modules'),
        ('Retrieval augmentation', 'add external evidence'),
    ],
    columns=['Method', 'Adaptation mechanism'],
)

model_uses

# 2. Causal Language Modeling

A causal language model estimates:


\[
P(x_1, \ldots, x_T) = \prod_{t=1}^{T} P(x_t \mid x_{<t})
\]

Each token is predicted from earlier tokens only.

# 3. Prompting

A prompt supplies instructions, context, examples, or constraints without updating
the model's parameters.

In [ ]:
prompt_parts = pd.DataFrame(
    [
        ('Instruction', 'what the model should do'),
        ('Context', 'relevant information'),
        ('Examples', 'desired input-output pattern'),
        ('Constraints', 'format, length, scope'),
        ('Output cue', 'where generation begins'),
    ],
    columns=['Prompt part', 'Purpose'],
)

prompt_parts

# 4. Generation Loop

Text generation repeatedly:

1. encodes the current sequence;
2. computes next-token logits;
3. transforms logits into a token choice;
4. appends the selected token;
5. stops when a criterion is met.

# 5. Greedy Decoding

Greedy decoding selects the highest-probability token at every step.

# 6. Temperature

Temperature rescales logits before softmax:

\[
p_i = \operatorname{softmax}(z_i / T)
\]

Lower temperature makes the distribution sharper. Higher temperature increases
randomness.

In [ ]:
logits_example = torch.tensor([2.0, 1.0, 0.2])

temperature_table = []

for temperature in [0.5, 1.0, 1.5]:
    probabilities = torch.softmax(
        logits_example / temperature,
        dim=0,
    )

    temperature_table.append(
        {
            'temperature': temperature,
            'p0': float(probabilities[0]),
            'p1': float(probabilities[1]),
            'p2': float(probabilities[2]),
        }
    )

pd.DataFrame(temperature_table)

# 7. Top-k Sampling

Top-k sampling retains only the `k` highest-scoring tokens before sampling.

# 8. Top-p Sampling

Nucleus or top-p sampling retains the smallest token set whose cumulative
probability reaches a threshold `p`.

# 9. Repetition Penalties

Repetition control can discourage previously generated tokens or repeated n-grams.

# 10. Stopping Criteria

Common stopping conditions:

- end-of-sequence token;
- maximum new-token count;
- custom stop sequence;
- task-specific structural completion.

# 11. Offline Command Corpus

The local experiment models controlled command sequences. This keeps execution
fast while exposing the full generation pipeline.

In [ ]:
subjects = [
    'robot',
    'drone',
    'assistant',
    'system',
]

actions = [
    'moves',
    'checks',
    'tracks',
    'reports',
    'opens',
    'closes',
]

objects = [
    'target',
    'status',
    'door',
    'route',
    'signal',
    'battery',
]

modifiers = [
    'carefully',
    'quickly',
    'safely',
    'again',
]

sentences = []

for subject in subjects:
    for action in actions:
        for obj in objects:
            for modifier in modifiers:
                sentences.append(
                    f'{subject} {action} {obj} {modifier}'
                )

print('Corpus size:', len(sentences))
sentences[:8]

# 12. Train, Validation, and Test Splits

In [ ]:
random.seed(42)
random.shuffle(sentences)

train_end = int(0.75 * len(sentences))
validation_end = int(0.875 * len(sentences))

train_texts = sentences[:train_end]
validation_texts = sentences[train_end:validation_end]
test_texts = sentences[validation_end:]

pd.Series(
    {
        'training': len(train_texts),
        'validation': len(validation_texts),
        'test': len(test_texts),
    }
)

# 13. Tokenization and Vocabulary

In [ ]:
TOKEN_PATTERN = re.compile(
    r"\b\w+(?:[-']\w+)*\b",
    flags=re.UNICODE,
)


def tokenize(text: str) -> list[str]:
    return TOKEN_PATTERN.findall(text.lower())


PAD_TOKEN = '<PAD>'
UNK_TOKEN = '<UNK>'
BOS_TOKEN = '<BOS>'
EOS_TOKEN = '<EOS>'

counts = Counter(
    token
    for text in train_texts
    for token in tokenize(text)
)

vocabulary = [
    PAD_TOKEN,
    UNK_TOKEN,
    BOS_TOKEN,
    EOS_TOKEN,
] + sorted(counts)

token2id = {
    token: index
    for index, token in enumerate(vocabulary)
}

id2token = {
    index: token
    for token, index in token2id.items()
}

PAD_ID = token2id[PAD_TOKEN]
UNK_ID = token2id[UNK_TOKEN]
BOS_ID = token2id[BOS_TOKEN]
EOS_ID = token2id[EOS_TOKEN]


def encode(text: str) -> list[int]:
    return [BOS_ID] + [
        token2id.get(token, UNK_ID)
        for token in tokenize(text)
    ] + [EOS_ID]


def decode(token_ids) -> str:
    tokens = []

    for token_id in token_ids:
        token = id2token[int(token_id)]

        if token == EOS_TOKEN:
            break

        if token not in {PAD_TOKEN, BOS_TOKEN}:
            tokens.append(token)

    return ' '.join(tokens)


print('Vocabulary size:', len(vocabulary))
encode(train_texts[0])

# 14. Dataset and Dynamic Padding

In [ ]:
class LanguageModelDataset(Dataset):
    def __init__(self, texts):
        self.texts = list(texts)

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, index):
        return torch.tensor(
            encode(self.texts[index]),
            dtype=torch.long,
        )


def collate_language_batch(batch):
    maximum_length = max(len(item) for item in batch)

    input_ids = torch.full(
        (len(batch), maximum_length),
        PAD_ID,
        dtype=torch.long,
    )

    for row, item in enumerate(batch):
        input_ids[row, :len(item)] = item

    return {
        'input_ids': input_ids,
        'padding_mask': input_ids == PAD_ID,
    }


train_dataset = LanguageModelDataset(train_texts)
validation_dataset = LanguageModelDataset(validation_texts)
test_dataset = LanguageModelDataset(test_texts)

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    collate_fn=collate_language_batch,
    generator=torch.Generator().manual_seed(42),
)

validation_loader = DataLoader(
    validation_dataset,
    batch_size=32,
    shuffle=False,
    collate_fn=collate_language_batch,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False,
    collate_fn=collate_language_batch,
)

sample_batch = next(iter(train_loader))
sample_batch['input_ids'].shape

# 15. Decoder-Only Transformer

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, model_dimension: int, maximum_length: int = 64):
        super().__init__()

        encoding = torch.zeros(maximum_length, model_dimension)
        positions = torch.arange(
            maximum_length,
            dtype=torch.float32,
        ).unsqueeze(1)

        rates = torch.exp(
            torch.arange(
                0,
                model_dimension,
                2,
                dtype=torch.float32,
            )
            * (-math.log(10000.0) / model_dimension)
        )

        encoding[:, 0::2] = torch.sin(positions * rates)
        encoding[:, 1::2] = torch.cos(positions * rates)

        self.register_buffer(
            'encoding',
            encoding.unsqueeze(0),
        )

    def forward(self, embeddings: torch.Tensor) -> torch.Tensor:
        return embeddings + self.encoding[:, :embeddings.size(1), :]


class DecoderOnlyTransformer(nn.Module):
    def __init__(
        self,
        vocabulary_size: int,
        model_dimension: int = 48,
        head_count: int = 4,
        layer_count: int = 2,
        feed_forward_dimension: int = 96,
        dropout: float = 0.10,
    ):
        super().__init__()

        self.model_dimension = model_dimension

        self.embedding = nn.Embedding(
            vocabulary_size,
            model_dimension,
            padding_idx=PAD_ID,
        )

        self.position = PositionalEncoding(model_dimension)

        layer = nn.TransformerEncoderLayer(
            d_model=model_dimension,
            nhead=head_count,
            dim_feedforward=feed_forward_dimension,
            dropout=dropout,
            activation='gelu',
            batch_first=True,
            norm_first=True,
        )

        self.encoder = nn.TransformerEncoder(
            layer,
            num_layers=layer_count,
        )

        self.output_layer = nn.Linear(
            model_dimension,
            vocabulary_size,
        )

    def causal_mask(self, length: int, device: torch.device) -> torch.Tensor:
        return torch.triu(
            torch.ones(
                length,
                length,
                dtype=torch.bool,
                device=device,
            ),
            diagonal=1,
        )

    def forward(
        self,
        input_ids: torch.Tensor,
        padding_mask: torch.Tensor,
    ):
        embeddings = self.embedding(input_ids) * math.sqrt(self.model_dimension)
        embeddings = self.position(embeddings)

        hidden_states = self.encoder(
            embeddings,
            mask=self.causal_mask(
                input_ids.size(1),
                input_ids.device,
            ),
            src_key_padding_mask=padding_mask,
        )

        logits = self.output_layer(hidden_states)

        return {
            'logits': logits,
            'last_hidden_state': hidden_states,
        }


DEVICE = torch.device('cpu')
torch.manual_seed(42)

model = DecoderOnlyTransformer(
    vocabulary_size=len(vocabulary)
).to(DEVICE)

print(
    'Trainable parameters:',
    sum(parameter.numel() for parameter in model.parameters()),
)

# 16. Shape and Mask Inspection

In [ ]:
with torch.no_grad():
    shape_output = model(
        sample_batch['input_ids'][:, :-1].to(DEVICE),
        sample_batch['padding_mask'][:, :-1].to(DEVICE),
    )

print('Logits:', shape_output['logits'].shape)

pd.DataFrame(
    model.causal_mask(6, DEVICE).int().cpu().numpy()
)

# 17. Training Utilities

In [ ]:
loss_function = nn.CrossEntropyLoss(
    ignore_index=PAD_ID
)


def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


def language_model_loss(
    logits: torch.Tensor,
    expected_ids: torch.Tensor,
) -> torch.Tensor:
    return loss_function(
        logits.reshape(-1, logits.size(-1)),
        expected_ids.reshape(-1),
    )


def evaluate_loss(model: nn.Module, loader: DataLoader) -> float:
    model.eval()
    losses = []

    with torch.no_grad():
        for batch in loader:
            full_ids = batch['input_ids'].to(DEVICE)
            input_ids = full_ids[:, :-1]
            expected_ids = full_ids[:, 1:]
            padding_mask = input_ids == PAD_ID

            output = model(input_ids, padding_mask)
            loss = language_model_loss(
                output['logits'],
                expected_ids,
            )
            losses.append(float(loss.item()))

    return float(np.mean(losses))

# 18. Model Training

In [ ]:
def train_model(
    model: nn.Module,
    epochs: int = 45,
    learning_rate: float = 0.003,
    patience: int = 8,
):
    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=learning_rate,
        weight_decay=1e-4,
    )

    best_state = copy.deepcopy(model.state_dict())
    best_validation_loss = float('inf')
    without_improvement = 0
    history = []

    for epoch in range(epochs):
        model.train()
        training_losses = []
        gradient_norms = []

        for batch in train_loader:
            full_ids = batch['input_ids'].to(DEVICE)
            input_ids = full_ids[:, :-1]
            expected_ids = full_ids[:, 1:]
            padding_mask = input_ids == PAD_ID

            optimizer.zero_grad()
            output = model(input_ids, padding_mask)
            loss = language_model_loss(
                output['logits'],
                expected_ids,
            )

            loss.backward()
            gradient_norm = clip_grad_norm_(
                model.parameters(),
                max_norm=5.0,
            )
            optimizer.step()

            training_losses.append(float(loss.item()))
            gradient_norms.append(float(gradient_norm))

        validation_loss = evaluate_loss(
            model,
            validation_loader,
        )

        history.append(
            {
                'epoch': epoch,
                'training_loss': float(np.mean(training_losses)),
                'validation_loss': validation_loss,
                'validation_perplexity': math.exp(min(validation_loss, 20.0)),
                'gradient_norm': float(np.mean(gradient_norms)),
            }
        )

        if validation_loss < best_validation_loss - 1e-5:
            best_validation_loss = validation_loss
            best_state = copy.deepcopy(model.state_dict())
            without_improvement = 0
        else:
            without_improvement += 1

        if without_improvement >= patience:
            break

    model.load_state_dict(best_state)
    return model, pd.DataFrame(history)


set_seed(42)
trained_model, training_history = train_model(model)

print('Epochs completed:', len(training_history))
print(
    'Best validation perplexity:',
    round(training_history['validation_perplexity'].min(), 3),
)

# 19. Learning Curves

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(
    training_history['epoch'],
    training_history['training_loss'],
    label='Training loss',
)
plt.plot(
    training_history['epoch'],
    training_history['validation_loss'],
    label='Validation loss',
)
plt.xlabel('Epoch')
plt.ylabel('Cross-entropy loss')
plt.title('Language Model Learning Curves')
plt.legend()
plt.tight_layout()
plt.show()

# 20. Validation Perplexity

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(
    training_history['epoch'],
    training_history['validation_perplexity'],
)
plt.xlabel('Epoch')
plt.ylabel('Validation perplexity')
plt.title('Validation Perplexity')
plt.tight_layout()
plt.show()

# 21. Next-Token Prediction

In [ ]:
def next_token_distribution(
    model: nn.Module,
    prompt: str,
) -> pd.DataFrame:
    model.eval()
    prompt_ids = encode(prompt)[:-1]

    input_ids = torch.tensor(
        [prompt_ids],
        dtype=torch.long,
        device=DEVICE,
    )

    with torch.no_grad():
        output = model(
            input_ids,
            input_ids == PAD_ID,
        )

    probabilities = torch.softmax(
        output['logits'][0, -1],
        dim=0,
    )

    top_probabilities, top_ids = torch.topk(
        probabilities,
        k=min(8, len(vocabulary)),
    )

    return pd.DataFrame(
        {
            'token': [
                id2token[int(token_id)]
                for token_id in top_ids
            ],
            'probability': top_probabilities.cpu().numpy(),
        }
    )


next_token_distribution(
    trained_model,
    'drone tracks',
)

# 22. Greedy Generation

In [ ]:
def apply_repetition_penalty(
    logits: torch.Tensor,
    generated_ids: list[int],
    penalty: float,
) -> torch.Tensor:
    adjusted = logits.clone()

    if penalty <= 1.0:
        return adjusted

    for token_id in set(generated_ids):
        if adjusted[token_id] < 0:
            adjusted[token_id] *= penalty
        else:
            adjusted[token_id] /= penalty

    return adjusted


def sample_next_token(
    logits: torch.Tensor,
    strategy: str,
    temperature: float,
    top_k: int,
    top_p: float,
    generator: torch.Generator,
) -> tuple[int, float]:
    temperature = max(temperature, 1e-5)
    scaled = logits / temperature

    if strategy == 'greedy':
        probabilities = torch.softmax(scaled, dim=-1)
        token_id = int(torch.argmax(probabilities).item())
        return token_id, float(probabilities[token_id].item())

    if strategy == 'top_k':
        k = max(1, min(top_k, scaled.numel()))
        values, indices = torch.topk(scaled, k=k)
        probabilities = torch.softmax(values, dim=-1)
        selected = int(torch.multinomial(
            probabilities,
            num_samples=1,
            generator=generator,
        ).item())
        token_id = int(indices[selected].item())
        return token_id, float(probabilities[selected].item())

    if strategy == 'top_p':
        sorted_logits, sorted_indices = torch.sort(
            scaled,
            descending=True,
        )
        sorted_probabilities = torch.softmax(sorted_logits, dim=-1)
        cumulative = torch.cumsum(sorted_probabilities, dim=-1)

        keep = cumulative <= top_p
        keep[0] = True

        filtered_logits = sorted_logits[keep]
        filtered_indices = sorted_indices[keep]
        filtered_probabilities = torch.softmax(filtered_logits, dim=-1)

        selected = int(torch.multinomial(
            filtered_probabilities,
            num_samples=1,
            generator=generator,
        ).item())
        token_id = int(filtered_indices[selected].item())
        return token_id, float(filtered_probabilities[selected].item())

    probabilities = torch.softmax(scaled, dim=-1)
    token_id = int(torch.multinomial(
        probabilities,
        num_samples=1,
        generator=generator,
    ).item())
    return token_id, float(probabilities[token_id].item())


def generate_text(
    model: nn.Module,
    prompt: str,
    maximum_new_tokens: int = 8,
    strategy: str = 'greedy',
    temperature: float = 1.0,
    top_k: int = 5,
    top_p: float = 0.9,
    repetition_penalty: float = 1.0,
    stop_tokens: set[int] | None = None,
    seed: int = 42,
) -> dict:
    model.eval()

    prompt_tokens = tokenize(prompt)
    generated_ids = [BOS_ID] + [
        token2id.get(token, UNK_ID)
        for token in prompt_tokens
    ]

    generator = torch.Generator(device='cpu').manual_seed(seed)
    step_probabilities = []

    if stop_tokens is None:
        stop_tokens = {EOS_ID}

    with torch.no_grad():
        for _ in range(maximum_new_tokens):
            input_ids = torch.tensor(
                [generated_ids],
                dtype=torch.long,
                device=DEVICE,
            )

            output = model(
                input_ids,
                input_ids == PAD_ID,
            )

            logits = output['logits'][0, -1]
            logits = apply_repetition_penalty(
                logits,
                generated_ids,
                repetition_penalty,
            )

            next_id, probability = sample_next_token(
                logits,
                strategy=strategy,
                temperature=temperature,
                top_k=top_k,
                top_p=top_p,
                generator=generator,
            )

            generated_ids.append(next_id)
            step_probabilities.append(probability)

            if next_id in stop_tokens:
                break

    return {
        'text': decode(generated_ids),
        'token_ids': generated_ids,
        'step_probabilities': step_probabilities,
        'mean_probability': float(np.mean(step_probabilities))
        if step_probabilities else 0.0,
        'log_probability': float(
            np.sum(
                np.log(
                    np.maximum(step_probabilities, 1e-12)
                )
            )
        ),
    }


generate_text(
    trained_model,
    'drone tracks',
    strategy='greedy',
)

# 23. Temperature Sampling

In [ ]:
temperature_results = []

for temperature in [0.5, 1.0, 1.5]:
    result = generate_text(
        trained_model,
        'assistant',
        strategy='sample',
        temperature=temperature,
        seed=7,
    )

    temperature_results.append(
        {
            'temperature': temperature,
            'text': result['text'],
            'mean_probability': result['mean_probability'],
        }
    )

pd.DataFrame(temperature_results)

# 24. Top-k Sampling

In [ ]:
top_k_results = []

for k in [2, 4, 8]:
    result = generate_text(
        trained_model,
        'robot',
        strategy='top_k',
        temperature=1.0,
        top_k=k,
        seed=11,
    )

    top_k_results.append(
        {
            'top_k': k,
            'text': result['text'],
        }
    )

pd.DataFrame(top_k_results)

# 25. Nucleus Sampling

In [ ]:
top_p_results = []

for probability_mass in [0.6, 0.8, 0.95]:
    result = generate_text(
        trained_model,
        'system',
        strategy='top_p',
        temperature=1.0,
        top_p=probability_mass,
        seed=19,
    )

    top_p_results.append(
        {
            'top_p': probability_mass,
            'text': result['text'],
        }
    )

pd.DataFrame(top_p_results)

# 26. Repetition Control

In [ ]:
repetition_results = []

for penalty in [1.0, 1.2, 1.5]:
    result = generate_text(
        trained_model,
        'drone',
        strategy='top_p',
        top_p=0.9,
        repetition_penalty=penalty,
        seed=23,
    )

    repetition_results.append(
        {
            'penalty': penalty,
            'text': result['text'],
        }
    )

pd.DataFrame(repetition_results)

# 27. Stopping Rules

In [ ]:
stopping_summary = pd.DataFrame(
    [
        ('EOS', 'model emits end token'),
        ('Maximum length', 'hard token limit'),
        ('Custom token', 'task-specific termination'),
        ('Stop string', 'post-tokenization text rule'),
    ],
    columns=['Rule', 'Meaning'],
)

stopping_summary

# 28. Strategy Comparison

In [ ]:
strategy_settings = [
    ('greedy', 1.0, 5, 0.9),
    ('sample', 0.8, 5, 0.9),
    ('top_k', 1.0, 4, 0.9),
    ('top_p', 1.0, 5, 0.85),
]

strategy_rows = []

for strategy, temperature, k, p in strategy_settings:
    for seed in [1, 2, 3]:
        result = generate_text(
            trained_model,
            'assistant',
            strategy=strategy,
            temperature=temperature,
            top_k=k,
            top_p=p,
            repetition_penalty=1.2,
            seed=seed,
        )

        strategy_rows.append(
            {
                'strategy': strategy,
                'seed': seed,
                'text': result['text'],
                'mean_probability': result['mean_probability'],
                'log_probability': result['log_probability'],
            }
        )

strategy_frame = pd.DataFrame(strategy_rows)
strategy_frame

# 29. Diversity Metrics

In [ ]:
def distinct_n(texts: list[str], order: int) -> float:
    ngrams = []

    for text in texts:
        tokens = tokenize(text)
        ngrams.extend(
            tuple(tokens[index:index + order])
            for index in range(len(tokens) - order + 1)
        )

    if not ngrams:
        return 0.0

    return len(set(ngrams)) / len(ngrams)


diversity_rows = []

for strategy, group in strategy_frame.groupby('strategy'):
    texts = group['text'].tolist()
    diversity_rows.append(
        {
            'strategy': strategy,
            'distinct_1': distinct_n(texts, 1),
            'distinct_2': distinct_n(texts, 2),
        }
    )

pd.DataFrame(diversity_rows)

# 30. Length Analysis

In [ ]:
strategy_frame['token_count'] = strategy_frame['text'].map(
    lambda text: len(tokenize(text))
)

strategy_frame.groupby('strategy')['token_count'].describe()

# 31. Confidence and Sequence Scores

Raw sequence log probability usually favors shorter sequences. Length-normalized
scores can support comparison, but they are not correctness guarantees.

In [ ]:
strategy_frame['length_normalized_score'] = (
    strategy_frame['log_probability']
    / strategy_frame['token_count'].clip(lower=1) ** 0.7
)

strategy_frame[
    [
        'strategy',
        'text',
        'mean_probability',
        'length_normalized_score',
    ]
]

# 32. Failure Analysis

Common generation failures include:

- repetition;
- premature stopping;
- incoherent continuation;
- prompt copying;
- unsupported claims;
- unsafe or disallowed content;
- format violations.

In [ ]:
failure_register = pd.DataFrame(
    [
        ('Repetition', 'repetition penalty or n-gram constraint'),
        ('Premature EOS', 'length controls and better training'),
        ('Unsupported claim', 'grounding and verification'),
        ('Format failure', 'structured prompts and validation'),
        ('Unsafe output', 'policy filters and human review'),
    ],
    columns=['Failure', 'Response'],
)

failure_register

# 33. Hallucination and Grounding

Language-model probability measures plausibility under the model, not factual
truth. Grounded systems should connect generation to verified evidence and preserve
source attribution.

# 34. Prompt Injection and Unsafe Instructions

External text can contain instructions that conflict with the intended task.
Production systems should separate trusted instructions from untrusted content,
validate outputs, and apply authorization boundaries.

In [ ]:
safety_controls = pd.DataFrame(
    [
        ('Instruction hierarchy', 'separate trusted and untrusted text'),
        ('Input filtering', 'detect risky content'),
        ('Output validation', 'enforce format and policy'),
        ('Grounding', 'require supporting evidence'),
        ('Human review', 'approve high-impact outputs'),
    ],
    columns=['Control', 'Purpose'],
)

safety_controls

# 35. Optional Hugging Face Setup

In [ ]:
TRANSFORMERS_AVAILABLE = (
    importlib.util.find_spec('transformers') is not None
)

RUN_HUGGING_FACE_DEMOS = False
USE_LOCAL_FILES_ONLY = True

MODEL_ID = 'distilgpt2'

pd.Series(
    {
        'transformers installed': TRANSFORMERS_AVAILABLE,
        'run demos': RUN_HUGGING_FACE_DEMOS,
        'local files only': USE_LOCAL_FILES_ONLY,
        'model ID': MODEL_ID,
    }
)

# 36. Optional AutoTokenizer

In [ ]:
if TRANSFORMERS_AVAILABLE and RUN_HUGGING_FACE_DEMOS:
    from transformers import AutoTokenizer

    hf_tokenizer = AutoTokenizer.from_pretrained(
        MODEL_ID,
        local_files_only=USE_LOCAL_FILES_ONLY,
    )

    if hf_tokenizer.pad_token is None:
        hf_tokenizer.pad_token = hf_tokenizer.eos_token

    hf_inputs = hf_tokenizer(
        ['The system reports'],
        return_tensors='pt',
        padding=True,
    )

    print(hf_inputs['input_ids'].shape)
else:
    print('Optional AutoTokenizer demo skipped.')

# 37. Optional AutoModelForCausalLM

In [ ]:
if TRANSFORMERS_AVAILABLE and RUN_HUGGING_FACE_DEMOS:
    from transformers import AutoModelForCausalLM

    hf_model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        local_files_only=USE_LOCAL_FILES_ONLY,
    ).to('cpu')

    hf_model.eval()

    with torch.no_grad():
        hf_output = hf_model(**hf_inputs)

    print('Logits:', hf_output.logits.shape)
else:
    print('Optional AutoModelForCausalLM demo skipped.')

# 38. Optional Text-Generation Pipeline

In [ ]:
if TRANSFORMERS_AVAILABLE and RUN_HUGGING_FACE_DEMOS:
    from transformers import pipeline

    generator_pipeline = pipeline(
        task='text-generation',
        model=MODEL_ID,
        tokenizer=MODEL_ID,
        device=-1,
        model_kwargs={
            'local_files_only': USE_LOCAL_FILES_ONLY,
        },
    )

    output = generator_pipeline(
        'The system reports',
        max_new_tokens=20,
        do_sample=True,
        temperature=0.8,
        top_p=0.9,
        num_return_sequences=2,
        pad_token_id=generator_pipeline.tokenizer.eos_token_id,
    )

    print(output)
else:
    print('Optional text-generation pipeline skipped.')

# 39. GenerationConfig

Hugging Face generation settings can be stored in a `GenerationConfig` object.

In [ ]:
generation_config_template = '''
from transformers import GenerationConfig

generation_config = GenerationConfig(
    max_new_tokens=64,
    do_sample=True,
    temperature=0.8,
    top_p=0.9,
    repetition_penalty=1.1,
    eos_token_id=tokenizer.eos_token_id,
    pad_token_id=tokenizer.eos_token_id,
)

outputs = model.generate(
    **inputs,
    generation_config=generation_config,
)
'''

print(generation_config_template)

# 40. Arabic and Multilingual Considerations

Arabic generation is affected by:

- rich morphology;
- attached clitics;
- optional tashkeel;
- MSA and dialect variation;
- tokenizer fragmentation;
- right-to-left display;
- code-switching;
- factual and cultural evaluation coverage.

In [ ]:
arabic_examples = pd.DataFrame(
    [
        (
            'اُكْتُبْ جُمْلَةً عَنِ الْبَحْثِ الْعِلْمِيِّ.',
            'fully vocalized MSA prompt',
        ),
        (
            'وَسَيَكْتُبُونَهَا',
            'morphologically complex form',
        ),
        (
            'بِالْمَدْرَسَةِ',
            'clitic plus definite noun',
        ),
    ],
    columns=['Arabic text', 'Issue'],
)

arabic_examples

For fully vocalized Arabic tasks, tashkeel should remain in prompts, generated
outputs, evaluation, and postprocessing whenever it is part of the task definition.

In [ ]:
arabic_generation_checks = pd.DataFrame(
    [
        ('Tashkeel', 'preserve or document removal'),
        ('Clitics', 'inspect segmentation and attachment'),
        ('Variety', 'separate MSA and dialect analysis'),
        ('Code-switching', 'evaluate mixed-language prompts'),
        ('Safety', 'include culturally relevant test cases'),
    ],
    columns=['Check', 'Action'],
)

arabic_generation_checks

# 41. Reproducibility and Reporting

Report:

- model ID and revision;
- tokenizer and revision;
- prompt template;
- generation parameters;
- random seed;
- maximum new tokens;
- stop criteria;
- repetition controls;
- decoding strategy;
- evaluation data;
- diversity and quality metrics;
- software versions;
- hardware;
- safety and limitation analysis.

In [ ]:
reproducibility_metadata = pd.Series(
    {
        'training examples': len(train_texts),
        'validation examples': len(validation_texts),
        'test examples': len(test_texts),
        'vocabulary size': len(vocabulary),
        'device': str(DEVICE),
        'seed': 42,
        'python': platform.python_version(),
        'torch': torch.__version__,
        'transformers installed': TRANSFORMERS_AVAILABLE,
        'optional demos enabled': RUN_HUGGING_FACE_DEMOS,
    },
    name='Lesson 39 experiment',
)

reproducibility_metadata

# 42. Knowledge Check

1. What is causal language modeling?
2. How does prompting differ from fine-tuning?
3. What does greedy decoding select?
4. How does temperature affect probabilities?
5. What does top-k sampling retain?
6. What does top-p sampling retain?
7. Why use repetition penalties?
8. What are stopping criteria?
9. What does perplexity measure?
10. What does distinct-n measure?
11. Why does log probability favor short sequences?
12. Why is model confidence not factual correctness?
13. What is prompt injection?
14. Why record a checkpoint revision?
15. How can tashkeel affect Arabic generation?

# 43. Exercises

## Exercise 1 — Greedy Decoding

Implement greedy generation from raw logits.

## Exercise 2 — Temperature

Compare temperatures on the same prompt and seed.

## Exercise 3 — Top-k

Compare several `k` values.

## Exercise 4 — Top-p

Compare several nucleus thresholds.

## Exercise 5 — Repetition

Add a no-repeat n-gram constraint.

## Exercise 6 — Stopping Criteria

Stop generation after a custom phrase.

## Exercise 7 — Hugging Face Generation

Run AutoModelForCausalLM on CPU with a cached small checkpoint.

## Exercise 8 — Diversity

Calculate distinct-1, distinct-2, and average length.

## Exercise 9 — Grounding

Add retrieved evidence and require supported outputs.

## Exercise 10 — Arabic Generation

Evaluate fully vocalized MSA prompts and outputs.

## Challenge Exercises

1. Add beam search and length penalties.
2. Implement contrastive search.
3. Add structured JSON output validation.
4. Compare Arabic-specific and multilingual causal models.
5. Build a grounded generation benchmark with human evaluation.

# 44. Summary and Next Lesson

In this lesson:

- pretrained causal language models were connected to next-token prediction;
- prompts were decomposed into instructions, context, examples, and constraints;
- a complete CPU-only decoder model was trained;
- greedy, temperature, top-k, top-p, and repetition-controlled decoding were
  implemented;
- stopping criteria and sequence scoring were examined;
- diversity and length statistics compared decoding strategies;
- degeneration, hallucination, prompt injection, and unsafe output risks were
  analyzed;
- optional Hugging Face AutoTokenizer, AutoModelForCausalLM, pipeline, and
  GenerationConfig workflows were provided;
- Arabic morphology, clitics, multilingual variation, and tashkeel were integrated
  into the generation workflow.

## Next Lesson

**Lesson 40: Sequence-to-Sequence Generation with Pretrained Transformers**
introduces pretrained encoder-decoder models for summarization, translation,
paraphrasing, generation metrics, beam search, and controlled decoding.

# References

- Hugging Face Transformers documentation: text generation, generation strategies,
  AutoModelForCausalLM, pipelines, and GenerationConfig.
- Holtzman, A. et al. *The Curious Case of Neural Text Degeneration*.
- Vaswani, A. et al. *Attention Is All You Need*.
- Jurafsky, D., & Martin, J. H. *Speech and Language Processing*.